# SV2 / Edge tuần 9 — chỉ đọc export Git

Ngày xuất bản: 09/11/2026. Phạm vi: KV260 → gateway GW_TRUNG trên PC SV2.
Notebook không gọi SSH, BLE, worker, prepare hay exporter; không ghi vào raw.
Chọn kernel PC/WSL có NumPy/Matplotlib, Shift+Enter từng ô.
Mean <20 ms là tiêu chí báo cáo SV2; p95/max vẫn vượt, review nhóm còn chờ.


In [ ]:
from pathlib import Path
import csv, hashlib, json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'edge').is_dir() and (p / 'contracts').is_dir())
DATA = ROOT / 'results/week9/edge'
for line in (DATA / 'sha256sum_export.txt').read_text().splitlines():
    digest, name = line.split('  ', 1)
    path = (DATA / name).resolve()
    assert path.is_relative_to(DATA.resolve()), name
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, name
summary = json.loads((DATA / 'summary.json').read_text())
provenance = json.loads((DATA / 'provenance.json').read_text())
with (DATA / 'latency_samples.csv').open(newline='') as handle:
    samples = list(csv.DictReader(handle))
print('SHA-256 export: OK; số thông điệp:', len(samples))
print('Nguồn:', DATA)
print('Commit công bố source:', provenance['published_source_commit'])
print('W10 SHA-256:', provenance['archive']['sha256'])
print('Phạm vi:', summary['scope'])
print('Trạng thái:', summary['status'])
print('Review:', summary['group_acceptance'])


## Bảng độ trễ ứng dụng (ms)

Cận dưới/trên phụ thuộc mô hình đồng hồ affine khả thi, không phải khoảng tin cậy thống kê.
Std mẫu ddof=1, percentile nearest rank. Không loại ngoại lệ.


In [ ]:
lines = ['| Lượt | Gửi/nhận | Mean | Std | P95 | P99 | Max | Mean lower…upper |',
         '|---|---:|---:|---:|---:|---:|---:|---:|']
for kind, run in summary['runs'].items():
    stats, match = run['one_way_ms'], run['matching']
    lines.append(f"| {kind} | {match['sent']}/{match['received']} | "
                 f"{stats['mean']:.3f} | {stats['std']:.3f} | {stats['p95']:.3f} | "
                 f"{stats['p99']:.3f} | {stats['max']:.3f} | "
                 f"{run['lower_ms']['mean']:.3f}…{run['upper_ms']['mean']:.3f} |")
display(Markdown('\n'.join(lines)))
for kind, run in summary['runs'].items():
    print(kind, run['run_id'], 'phân loại 20 ms:', run['threshold_20ms'],
          'mean upper<20:', run['mean_upper_bound_lt_20ms'],
          'mọi upper<20:', run['all_upper_bounds_lt_20ms'])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), layout='constrained')
for ax, kind in zip(axes, ('idle', 'varying')):
    values = np.sort([float(row['t_one_way_midpoint_ms']) for row in samples if row['kind'] == kind])
    ax.plot(values, np.arange(1, len(values) + 1) / len(values))
    ax.axvline(20, color='tab:red', linestyle='--', label='20 ms')
    ax.axhline(.95, color='grey', linestyle=':', label='p95')
    ax.set(xscale='log', xlabel='Application one-way midpoint (ms)', ylabel='CDF', title=kind)
    ax.grid(alpha=.25)
    ax.legend()
plt.show()


## Queue, calibration và receive gaps

Service bao gồm dispatch/native CPU+DPU; ngưỡng MQTT 20 ms không áp vào service.
Hai lượt không có foreground BLE. Gaps không bị xóa và chưa xác định được nguyên nhân spike.


In [ ]:
for kind, run in summary['runs'].items():
    print('\n', kind, run['run_id'])
    print('Queue:', run['queue'])
    print('Service (ms):', run['service_ms'])
    print('Background wait (ms):', run['background_queue_wait_ms'])
    print('Calibration:', run['calibration'])
    print('Receive gaps >500 ms:', run['receive_gaps_over_500ms'])
    print('Giới hạn:', *run['measurement_limitations'], sep='\n  ')


## B5 lịch sử và nơi đọc tiếp

B5 độc lập phép đo MQTT tự thực hiện. Chỉ đọc đối chiếu đã lưu, không chạy lại PPK2.
Trạng thái pending trong raw Edge là lúc capture; checker Device công bố sau đó là nguồn review.


In [ ]:
b5 = json.loads((DATA / 'b5_review.json').read_text())
print('Manifest Device đã kiểm:', b5['device_manifest_files_verified'], 'file')
for phase, run in b5['runs'].items():
    print(phase, 'Device:', run['device_status'], 'Edge OK:', run['edge']['ok'],
          'logits bit match:', run['offline_logits_bit_matches'],
          'nguồn:', run['device_receipt'])
display(Markdown('[Báo cáo](../../docs/sv2_edge_week9_report.md) · '
                 '[Hồ sơ](../../results/week9/edge/README.md)'))
